In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import RandomOverSampler
import os

In [ ]:
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo

# fetch dataset
magic_gamma_telescope = fetch_ucirepo(id=159)

# data (as pandas dataframes)
X = magic_gamma_telescope.data.features
y = magic_gamma_telescope.data.targets

# metadata
print(magic_gamma_telescope.metadata)

# variable information
print(magic_gamma_telescope.variables)


In [ ]:
y_df = pd.DataFrame(y)
df = pd.concat([X, y_df], axis=1)


In [ ]:
df.head()

In [ ]:
df['class']=(df['class']=='g').astype(int)
y['class']=(y['class']=='g').astype(int)

In [ ]:
df.head()

In [ ]:
for label in df.columns:
  plt.hist(df[df['class']==1][label], color='blue', alpha=0.7, label='gamma',density=True)
  plt.hist(df[df['class']==0][label], color='red', alpha=0.7, label='hadron',density=True)
  plt.xlabel(label)
  plt.ylabel('probability')
  plt.legend()
  plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.25, random_state=42
)

In [ ]:
train = pd.concat([X_train, y_train], axis=1)
test = pd.concat([X_test, y_test], axis=1)
valid = pd.concat([X_valid, y_valid], axis=1)



In [ ]:
def scale(df,oversample=False):

  X = df.drop('class', axis=1)
  y = df['class']
  scaler = StandardScaler()
  X = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
  data= np.hstack((X, np.reshape(y,(len(y), 1))))
  if oversample:
    ros = RandomOverSampler(random_state=42)
    X, y = ros.fit_resample(X, y)
  return data,X,y

In [ ]:
train

In [ ]:
train = train.reset_index(drop=True)
valid = valid.reset_index(drop=True)
test = test.reset_index(drop=True)

In [ ]:
train, X_train, y_train = scale(train,oversample=True)
valid, X_valid, y_valid = scale(valid,oversample=False)
test, X_test, y_test = scale(test,oversample=False)

In [ ]:
y_train

In [ ]:
sum(y_train)/len(y_train)


# Naive Bayes Classifier

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

In [ ]:
nb_model=GaussianNB()
nb_model.fit(X_train,y_train)

In [ ]:
y_pred=nb_model.predict(X_test)

In [ ]:
print (classification_report(y_test,y_pred))

In [ ]:
os.makedirs('../data', exist_ok=True)

# Extract column names
column_names = df.columns.tolist()

# Save into the data folder
pd.DataFrame(test, columns=column_names).to_csv('../data/test.csv', index=False)
pd.DataFrame(train, columns=column_names).to_csv('../data/train.csv', index=False)
pd.DataFrame(valid, columns=column_names).to_csv('../data/valid.csv', index=False)